# 07 — Trabajo con JSON

## Objetivo
Entender la estructura JSON y parsear respuestas del LLM con el módulo `json` de Python.

## Conceptos
- JSON como formato de intercambio de datos
- Diccionarios y listas anidadas
- **`json.loads()`**: convertir string JSON → objeto Python
- Pedir al LLM respuestas en JSON estructurado

## Dependencia
Continúa el trabajo con reseñas negativas del notebook **06**.


In [1]:
import sys
sys.path.insert(0, "..")

from notebooks._setup import load_env, require_key, DATOS_DIR, OUTPUT_DIR, safe_generate, use_real_apis, LocalGeminiClient, LocalGroqClient

load_env()
OUTPUT_DIR.mkdir(exist_ok=True)

require_key("GROQ_API_KEY")
import pandas as pd

if use_real_apis():
    from groq import Groq
    client_groq = Groq()
else:
    client_groq = LocalGroqClient()
reviews_path = OUTPUT_DIR / "reviews_sentimiento.csv"
if reviews_path.exists():
    df_reviews = pd.read_csv(reviews_path)
else:
    df_reviews = pd.read_csv(DATOS_DIR / "reviews.csv")
    df_reviews["Sentimiento"] = ["Negativa" if i % 3 == 0 else "Positiva" if i % 3 == 1 else "Neutra" for i in range(len(df_reviews))]
    df_reviews.to_csv(reviews_path, index=False)

df_reviews_negativas = df_reviews[df_reviews["Sentimiento"] == "Negativa"]
reviews_negativas = df_reviews_negativas["reviewText"]
reviews_negativas_unidas = "####".join(reviews_negativas)

categorias = safe_generate(lambda texto: texto, reviews_negativas_unidas, task="categorias")

In [2]:
categorias = safe_generate(
    lambda texto: texto,
    reviews_negativas_unidas,
    task="categorias",
)
categorias

'calidad del producto, entrega, precio, atencion al cliente'

In [3]:
lista_de_categorias = categorias.split(", ")
lista_de_categorias

['calidad del producto', 'entrega', 'precio', 'atencion al cliente']

# JSON

In [4]:
{
    "name": "Chris",
    "age": 23,
    "address": {
        "city": "New York",
        "country": "America"
    },
    "friends": [
        {
            "name": "Emily",
            "hobbies": [
                "biking",
                "music",
                "gaming"
            ]
        },
        {
            "name": "John",
            "hobbies": [
                "soccer",
                "gaming"
            ]
        }
    ]
}

{'name': 'Chris',
 'age': 23,
 'address': {'city': 'New York', 'country': 'America'},
 'friends': [{'name': 'Emily', 'hobbies': ['biking', 'music', 'gaming']},
  {'name': 'John', 'hobbies': ['soccer', 'gaming']}]}

In [5]:
prompt = f"""
Eres un analista de datos. Clasifica estas resenas negativas separadas por ####.
Devuelve SOLO un JSON array con objetos que tengan categoria y resenas.

{reviews_negativas_unidas}
"""
json_de_reviews_clasificadas = safe_generate(
    lambda texto: client_groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": texto}],
    ).choices[0].message.content,
    prompt,
    task="json_reviews",
)
print(json_de_reviews_clasificadas)

[{"categoria": "calidad del producto", "resenas": ["Producto con fallas o baja durabilidad"]}, {"categoria": "entrega", "resenas": ["Demoras o problemas con el envio"]}, {"categoria": "atencion al cliente", "resenas": ["Dificultad para resolver reclamos"]}]


In [6]:
import json

dic_reviews_negativas_clasificadas = json.loads(json_de_reviews_clasificadas)
dic_reviews_negativas_clasificadas

[{'categoria': 'calidad del producto',
  'resenas': ['Producto con fallas o baja durabilidad']},
 {'categoria': 'entrega', 'resenas': ['Demoras o problemas con el envio']},
 {'categoria': 'atencion al cliente',
  'resenas': ['Dificultad para resolver reclamos']}]